# 05 - Validation receipt and reproducibility

## Objective

Run the same public Case twice through the CLI, inspect the persisted evidence, and learn which identity fields are deterministic and which timestamps are not.

## Source, assumptions, and units

The source is the bundled IEEE13 public demonstrator selected by `cept study demo load-flow`. The Case fingerprint identifies the typed input; solver voltage magnitudes are pu, and the receipt identifies OpenDSS and the installed public version. No field measurements or project-specific settings are supplied.

## Prediction

Two runs with the same Case and solver should have the same Case fingerprint and load-flow payload, while attempt identity and creation timestamps may differ. Both exact run directories should verify with the public claim `WORKFLOW_VALIDATED`.

## Action

Stream two `cept study demo load-flow` commands to two explicit run paths, then stream `cept study verify` for each path. No `latest` directory or modification time is used to select evidence.

## Verification

Read `case.json`, `results.json`, and `public-verification.json` from both named runs. Compare the Case and load-flow payloads, then assert the exact verification receipts.

## Interpretation

A verification receipt proves the persisted public artifact set is internally consistent for its bounded workflow. It does not turn a demonstrator into field evidence, independent reference agreement, or `PROJECT_VALIDATED`.

## Exercise

Run the same cells after changing the exact output directory names, then change one Case input in a lesson that owns an inline Case. Predict which fingerprint and result fields should change before rerunning.

## Runtime requirements

Use Python 3.10 or newer with an existing installed `cept` command, or provide a caller-owned wheel through `CEPT_WHEEL_URL` and its exact `CEPT_WHEEL_SHA256`. The wheel must provide CEPT and OpenDSS. No released PyPI version is assumed. Jupyter is needed only to execute the notebook.

In [ ]:
import hashlib
import importlib.util
import json
import os
import shlex
import subprocess
import sys
import urllib.parse
import urllib.request
from pathlib import Path

DEFAULT_WHEEL_URL = 'https://github.com/sarutesri/cept-studio-edu/releases/download/v0.2.0-edu.1/cept_power_studio-0.2.0.dev0-py3-none-any.whl'
DEFAULT_WHEEL_SHA256 = 'c7e609a1d9cc85b322bfb615f0c796c7ea5c43815b197289eb555786964478bc'
configured_url = os.environ.get('CEPT_WHEEL_URL')
WHEEL_URL = (DEFAULT_WHEEL_URL if configured_url is None and importlib.util.find_spec('cept') is None else (configured_url or '')).strip()
WHEEL_SHA256 = os.environ.get('CEPT_WHEEL_SHA256', DEFAULT_WHEEL_SHA256 if WHEEL_URL == DEFAULT_WHEEL_URL else '').strip().lower()
if WHEEL_URL:
    if len(WHEEL_SHA256) != 64 or any(character not in '0123456789abcdef' for character in WHEEL_SHA256):
        raise ValueError('CEPT_WHEEL_SHA256 must be the caller-provided 64-character SHA-256')
    wheel_path = Path.cwd() / Path(urllib.parse.urlparse(WHEEL_URL).path).name
    print(f'Downloading caller-provided wheel: {WHEEL_URL}')
    urllib.request.urlretrieve(WHEEL_URL, wheel_path)
    digest = hashlib.sha256(wheel_path.read_bytes()).hexdigest()
    if digest != WHEEL_SHA256:
        raise ValueError(f'wheel hash mismatch: expected {WHEEL_SHA256}, got {digest}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', str(wheel_path)], check=True)
else:
    print('CEPT_WHEEL_URL not supplied; using the existing installed environment.')
CLI_BIN = Path(sys.executable).parent / ('cept.exe' if os.name == 'nt' else 'cept')
if not CLI_BIN.is_file():
    raise RuntimeError('cept launcher not found next to Python; reinstall the pinned public wheel')
CLI = str(CLI_BIN)
print('CLI: cept --version')
print(subprocess.run([CLI, '--version'], capture_output=True, text=True, check=True).stdout.strip())
def run_cli(*arguments):
    display = 'cept ' + shlex.join([str(argument) for argument in arguments])
    command = [CLI, *[str(argument) for argument in arguments]]
    print('$ ' + display, flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=Path.cwd())
    lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    returncode = process.wait()
    output = ''.join(lines)
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, ['cept', *[str(argument) for argument in arguments]], output=output)
    return json.loads(output) if output.strip().startswith('{') else output

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def show_table(headers, rows):
    print('| ' + ' | '.join(headers) + ' |')
    print('| ' + ' | '.join('---' for _ in headers) + ' |')
    for row in rows:
        print('| ' + ' | '.join(str(value) for value in row) + ' |')


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version


cept-power-studio 0.2.0.dev0


In [ ]:
RUN_ONE = Path.cwd() / 'runs' / '05-reproducibility-1'
RUN_TWO = Path.cwd() / 'runs' / '05-reproducibility-2'
first_summary = run_cli('study', 'demo', 'load-flow', '--network', 'ieee13', '--out', RUN_ONE, '--force')
second_summary = run_cli('study', 'demo', 'load-flow', '--network', 'ieee13', '--out', RUN_TWO, '--force')
first_verify = run_cli('study', 'verify', RUN_ONE)
second_verify = run_cli('study', 'verify', RUN_TWO)
case_one = read_json(RUN_ONE / 'case.json')
case_two = read_json(RUN_TWO / 'case.json')
result_one = read_json(RUN_ONE / 'results.json')
result_two = read_json(RUN_TWO / 'results.json')
receipt_one = read_json(RUN_ONE / 'public-verification.json')
receipt_two = read_json(RUN_TWO / 'public-verification.json')
show_table(['run', 'case fingerprint', 'study type', 'engine', 'claim', 'verified'], [(str(RUN_ONE), receipt_one['case_fingerprint'], receipt_one['study_type'], receipt_one['engine'], receipt_one['claim'], first_verify['passed']), (str(RUN_TWO), receipt_two['case_fingerprint'], receipt_two['study_type'], receipt_two['engine'], receipt_two['claim'], second_verify['passed'])])
show_table(['run', 'bus', 'phase', 'voltage magnitude', 'unit'], [(str(RUN_ONE), row['bus'], row['phase'], row['v_pu'], 'pu') for row in result_one['load_flow']['bus_voltages']])
assert first_summary['status'] == 'PASS' and second_summary['status'] == 'PASS'
assert first_verify['passed'] is True and second_verify['passed'] is True
assert case_one == case_two
assert result_one['case_fingerprint'] == result_two['case_fingerprint']
assert result_one['load_flow'] == result_two['load_flow']
assert receipt_one['claim'] == 'WORKFLOW_VALIDATED' and receipt_two['claim'] == 'WORKFLOW_VALIDATED'
assert receipt_one['attempt_id'] != receipt_two['attempt_id']


$ cept study demo load-flow --network ieee13 --out '<notebook-workspace>\runs\05-reproducibility-1' --force


{


  "status": "PASS",


  "claim": "WORKFLOW_VALIDATED",


  "study_type": "load_flow",


...

  "schema": "cept-public-verification-v1",


  "status": "PASS",


  "study_type": "load_flow"


}


| run | case fingerprint | study type | engine | claim | verified |
| --- | --- | --- | --- | --- | --- |
| <notebook-workspace>\runs\05-reproducibility-1 | 748c8026c9d6 | load_flow | opendss | WORKFLOW_VALIDATED | True |
| <notebook-workspace>\runs\05-reproducibility-2 | 748c8026c9d6 | load_flow | opendss | WORKFLOW_VALIDATED | True |
| run | bus | phase | voltage magnitude | unit |
| --- | --- | --- | --- | --- |
| <notebook-workspace>\runs\05-reproducibility-1 | sourcebus | 1 | 0.999974 | pu |
| <notebook-workspace>\runs\05-reproducibility-1 | sourcebus | 2 | 0.999994 | pu |
| <notebook-workspace>\runs\05-reproducibility-1 | sourcebus | 3 | 0.99995 | pu |
| <notebook-workspace>\runs\05-reproducibility-1 | 650 | 1 | 0.999911 | pu |
| <notebook-workspace>\runs\05-reproducibility-1 | 650 | 2 | 0.999971 | pu |
| <notebook-workspace>\runs\05-reproducibility-1 | 650 | 3 | 0.999931 | pu |
| <notebook-workspace>\runs\05-reproducibility-1 | rg60 | 1 | 1.056033 | pu |
| <notebook-workspace>\r

The comparison uses actual persisted Case and solver payloads. Attempt IDs and timestamps identify separate executions; they are not used to choose which result is authoritative. Keep both exact run paths and their public verification receipts when sharing this exercise.